# Shaft Connections, Lab 1: Key <br>
![](images/Lab_Key_Design.png)
 <br><br>
For a long drive shaft with a $ T_{nom} = 450 Nm $ , a flange coupler (Dutch: flenskopelling) is being used to connect to the input of a gearbox. <br>
The DIN 116-A60 flange is made out of EN-GJS-450, the shafts material is E295.<br>
Since impact load is expected, a $ K_A = 1.5 $ is to be considered. <br>
- Verify if a key, DIN 6885 type A, can be used
- Specify the desired ISO-tolerance classes for both shaft and hole

In [1]:
# import statements
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt

import MechDesign.Helpers as HM

from MechDesign.Units.Units import m_, mm_, kg_, s_, N_, rpm_, W_, deg_
import MechDesign.Units.UnitMethods as UM

import MechDesign.RnM as RnM

In [2]:
# creating a new shaft connection, (this is 1 specific connection in your design)
SC = RnM.ShaftConnection() 

# setting up constants, based on assignment (units are specified using a trailing '_')
SC.d = 60*mm_ 
SC.T_nom = 450*N_*m_
SC.K_A = 1.5

# setting up constants that are not likely to change
SC.phi = 1 # 1key
SC.n = 1 # 1 key
SC.K_lambda = 1 # method C

SC.K_t = 0.92
SC.b = 18*mm_
SC.h = 11*mm_

# setting up constants chosen more arbitrary
# guessing a first value for key length
SC.l = 80*mm_  # when using l'<=1.3*d with a shaft of 60mm and key width of 18 mm l'<= 78 and l<78+18=96mm and a DIN 116 A60 has max length of 85mm

# setting up helper functions
SC.lprime = SC.E12_1_hI_KeyEffectiveLength()
SC.hprime = SC.E12_1_hJ_KeyEffectiveHeight()
SC.T_eq = SC.E12_1_hC_DynamicLoadTorque()

# setting up the main equations
SC.p_gem = SC.E12_1B_KeyAveragePressure()
HM.EqPrint("p_gem",SC.p_gem)
# convert units, the previous expression still has both m_ and mm_
subst = {m_:1e3*mm_}
SC.p_gem = SC.p_gem.evalf(subs=subst)  # this line will substitute the symbol 'm_' with '1e3*mm_
t=HM.EqPrint('p_gem',SC.p_gem)

Eq(p_gem, 0.07331*N_*m_/mm_**3)

Eq(p_gem, 73.31*N_/mm_**2)

---
  
### finding the relevant limits
using method C, we will find the relevant limits for key, shaft and hub

In [3]:
# getting relevant limits for the surface pressure
R_eNShaft = 295* N_/mm_**2
K_tShaft = 0.92
R_eNKey = 420* N_/mm_**2
K_tKey = K_tShaft
R_mNHub = 450* N_/mm_**2
K_tHub = 1
# calculate the reference strengths
R_eShaft = R_eNShaft * K_tShaft
R_eKey = R_eNKey * K_tKey
R_mHub = R_mNHub  * K_tHub 

# display the results
HM.EqPrint('R_eShaft',R_eShaft)
HM.EqPrint('R_eKey',R_eKey)
HM.EqPrint('R_mHub',R_mHub)

# calculate overall safety
S = min(UM.RemoveUnits(R_eShaft),UM.RemoveUnits(R_eKey),UM.RemoveUnits(R_mHub))*N_/mm_**2 / SC.p_gem
t=HM.EqPrint('S',S)

Eq(R_eShaft, 271.4*N_/mm_**2)

Eq(R_eKey, 386.4*N_/mm_**2)

Eq(R_mHub, 450*N_/mm_**2)

Eq(S, 3.702)

---
  
### Conclusion:
the average pressure is well below the strength limit of the weakest material.  The design is safe
<br><br>
***

Below a set of images can be found that could serve as inspiration to create your own graphs. 


<div>
<img src="images/KeyFigExample1.PNG" width="700"/>
<img src="images/KeyFigExample2.PNG" width="700"/>
<img src="images/KeyFigExample3.PNG" width="700"/>
</div>

For a more comprehensive analysis, the key's dimensions incrementally correlate with the shaft's diameter. This can be accomplished by employing:  

```
# the height and width are depending on the diameter.
SC.b = sp.Piecewise((14*mm_,SC.d<=50),(16*mm_,SC.d<=58),(18*mm_,SC.d<=65))
SC.h = sp.Piecewise((9*mm_,SC.d<=50),(10*mm_,SC.d<=58),(11*mm_,SC.d<=65))
```